# Alakoro FiberSense v2.11.0 — Demonstração da Interface Gráfica
# Alakoro FiberSense v2.11.0 — Graphical User Interface Demo

**Autor/Author:** Luiz Paulo Colombiano | **Licença/License:** MIT | **Versão/Version:** 2.10.0

Este notebook apresenta a interface desktop do Alakoro FiberSense construída com **PySide6** e **PyQtGraph**.
This notebook showcases the Alakoro FiberSense desktop interface built with **PySide6** and **PyQtGraph**.

---

## 1. Instalação / Installation

A GUI é um extra opcional. Instale com:
The GUI is an optional extra. Install it with:

```bash
pip install alakoro-fibersense[gui]
```

Ou execute esta célula / Or run this cell:

In [ ]:
# Instalar a versão com suporte a GUI / Install the GUI-enabled version
!pip install alakoro-fibersense[gui]

## 2. Lançar a Interface / Launch the Interface

Após a instalação, basta executar no terminal:
After installation, simply run in the terminal:

```bash
alakoro-gui
```

Equivalente em Python / Python equivalent:

In [ ]:
from src.gui.main_window import main

# Inicia o loop de eventos Qt / Starts the Qt event loop
main()

## 3. Tela Inicial — Carregamento e Heatmap / Main Screen — Loading and Heatmap

A janela principal é dividida em duas regiões:
The main window is split into two regions:

- **Esquerda / Left:** abas de visualização (heatmap 2D, perfis e espectrograma).
- **Direita / Right:** painéis de processadores (filters, thermal, ML/validate, presets).

Recursos rápidos / Quick features:
- **Drag-and-drop:** arraste um arquivo DFOS diretamente na janela.
- **Arquivos recentes:** menu `Arquivo → Recentes`.
- **Undo/Redo:** menu `Editar` ou barra de ferramentas.
- **Cursor sync:** passe o mouse no heatmap para ver `(t, ch, amp)` na barra de status; clique para fixar o perfil naquele tempo.

Use **Arquivo → Abrir** para carregar TDMS, SEG-Y, NetCDF, DASDAE ou dados sintéticos.
Use **File → Open** to load TDMS, SEG-Y, NetCDF, DASDAE or synthetic data.

In [ ]:
from IPython.display import Image, display
display(Image(filename='images/gui_main_window.png'))

### Equivalente em código — carregar dados / Code equivalent — load data

```python
from src.io.alakoro_spool import AlakoroPatch
from src.io.dasdae import DASDAEAdapter
import numpy as np

data = np.random.randn(512, 64)  # (time, channels)
dc_patch = DASDAEAdapter.array_to_patch(
    data, modality='das', dt_s=0.001, dx_m=1.0
)
patch = AlakoroPatch(dc_patch, modality='das')
```

## 4. Visualização de Perfis / Profile View

A aba **Profiles** mostra múltiplos perfis de amplitude em tempos selecionados.
Você pode marcar/desmarcar tempos, adicionar novos via spin box e visualizar
a média ± desvio padrão.
The **Profiles** tab shows multiple amplitude profiles at selected times.
You can check/uncheck times, add new ones via spin box and view mean ± std.

In [ ]:
display(Image(filename='images/gui_profile_view.png'))

### Equivalente em código — perfis / Code equivalent — profiles

```python
from src.gui.viewers.profile_view import ProfileView
from PySide6.QtWidgets import QApplication

app = QApplication([])
viewer = ProfileView()
viewer.set_data(data, times=[0, 128, 256, 384, 511])
viewer.show()
app.exec()
```

## 5. Espectrograma / Spectrogram

A aba **Spectrogram** calcula STFT do canal selecionado com parâmetros configuráveis.
The **Spectrogram** tab computes STFT for the selected channel with configurable parameters.

In [ ]:
display(Image(filename='images/gui_spectrogram_view.png'))

### Equivalente em código — espectrograma / Code equivalent — spectrogram

```python
from scipy.signal import stft

channel = 0
f, t, Zxx = stft(data[:, channel], fs=1000.0, nperseg=256, noverlap=128)
magnitude = np.abs(Zxx)
```

## 6. Painel ML / Validate / ML / Validate Panel

- Rode o `SignatureValidator` e veja o relatório detalhado.
- Aplique máscara de anomalias com threshold ajustável.
- Carregue modelo treinado e execute inferência.
- Use o **Train Model Wizard** para treinar Random Forest/SVM a partir de features `.npy`.

Run `SignatureValidator`, view detailed report, apply anomaly mask, load trained model,
or use the **Train Model Wizard** to train Random Forest/SVM from `.npy` features.

In [ ]:
display(Image(filename='images/gui_ml_panel.png'))

## 7. Presets e Pipeline Visual / Presets and Visual Pipeline

Monte pipelines de processamento visualmente, salve como JSON em `~/.alakoro/presets`
e aplique em lote via **Batch Processing**.
Build processing pipelines visually, save as JSON in `~/.alakoro/presets`
and apply in batch via **Batch Processing**.

In [ ]:
display(Image(filename='images/gui_presets_panel.png'))

### Equivalente em código — preset / Code equivalent — preset

```python
pipeline = [
    {'action': 'demean', 'kwargs': {}},
    {'action': 'butterworth_bandpass', 'kwargs': {'low_hz': 1, 'high_hz': 100}},
    {'action': 'svd_denoise', 'kwargs': {'n_components': 5}},
]

from src.gui.main_window import AlakoroMainWindow
window = AlakoroMainWindow()
window._run_preset_pipeline(pipeline)
```

## 8. Painel de Processadores / Processor Panel

Os processadores estão organizados em abas laterais:
Processors are organized into side tabs:

| Aba / Tab | Funcionalidades / Features |
|---|---|
| **Filters** | Butterworth lowpass/highpass/bandpass, detrend, demean, taper, median 1D, SVD denoise, STA/LTA, Hilbert envelope, PSD |
| **Thermal** | Thermal gradient, baseline correction, LF-DAS/DTS pipeline |
| **ML/Validate** | SignatureValidator, anomaly mask, load trained model, inference, train wizard |
| **Presets** | Visual pipeline editor, save/load JSON presets, batch processing |

### Equivalente em código — filtro Butterworth / Code equivalent — Butterworth filter

```python
from src.processing.advanced_filters import ButterworthFilter

filt = ButterworthFilter(fs=1000.0, lowcut=10.0, highcut=100.0, order=4)
filtered = filt.bandpass(patch.data)
```

### Equivalente em código — validação / Code equivalent — validation

```python
from src.simulation import WellGeometry, AcquisitionConfig
from src.validation import SignatureValidator

well = WellGeometry(depth_top=0, depth_bottom=64, n_channels=64)
acq = AcquisitionConfig(
    sampling_rate_hz=1000.0,
    trace_interval_s=2.0,
    duration_s=patch.shape[0] * 2.0
)
validator = SignatureValidator(well, acq)
result = validator.validate_signature(signature_dict, lfdas_result)
print(f'{result["passed"]}/{result["total"]} passaram — {result["success_rate"]:.0f}%')
```

## 9. Pipeline LF-DAS / LF-DAS Pipeline

O painel **Thermal** permite configurar o pipeline LF-DAS/DTS, definindo
cutoff, taxa de atualização e coeficiente térmico.
The **Thermal** panel lets you configure the LF-DAS/DTS pipeline, setting
cutoff, refresh rate and thermal coefficient.

In [ ]:
from src.processing import LFDASProcessor

lfdas = LFDASProcessor(
    cutoff_hz=1.0,
    refresh_rate_target_s=2.0,
    thermal_coefficient=1.0e-5
)
result = lfdas.process(patch.data, trace_interval_s=2.0)

print(f'Temperatura estimada / Estimated temperature: {result["temperature"].shape}')
print(f'Taxa de atualização / Refresh rate: {result["refresh_rate_s"]:.2f}s')

## 10. Exportação e Relatórios / Export and Reports

Use **Arquivo → Salvar** para exportar o resultado atual em NetCDF, NumPy, PNG, CSV, Avro ou Protobuf.
Use **Arquivo → Exportar Figura** para salvar heatmap/perfil com tamanho, DPI e colormap configuráveis.
Use **Arquivo → Gerar Relatório** para criar HTML/PDF com metadados, visualização e validação.

Use **File → Save** to export current result as NetCDF, NumPy, PNG, CSV, Avro or Protobuf.
Use **File → Export Figure** to save heatmap/profile with configurable size, DPI and colormap.
Use **File → Generate Report** to create HTML/PDF with metadata, visualization and validation.

### Equivalente em código — exportar NetCDF / Code equivalent — export NetCDF

```python
import xarray as xr

xr.DataArray(
    patch.data,
    dims=('time', 'channel'),
    name='amplitude'
).to_netcdf('resultado.nc')
```

## 11. Dicas de Execução / Runtime Tips

- **Servidores sem display / Headless servers:** use `QT_QPA_PLATFORM=offscreen alakoro-gui`.
- **Janelas offscreen para testes:** execute a aplicação com `window.grab()` para capturar screenshots.
- **Alto DPI:** a interface responde a `QT_SCALE_FACTOR=1.5` se necessário.
- **Logs:** acesse via `Ajuda → Ver Log`. Arquivo salvo em `~/.alakoro/alakoro.log`.
- **Cache:** processamentos repetidos com mesmos dados/parâmetros usam cache automático.
- **Downsampling:** heatmaps grandes são automaticamente reduzidos para visualização fluida.

---

## 12. Próximos Passos / Next Steps

- Experimente carregar seus próprios arquivos TDMS/SEG-Y/NetCDF.
- Compare o resultado dos processadores avançados (Butterworth, FFT, wavelets).
- Crie presets reutilizáveis e aplique em lote em campanhas de aquisição.
- Integre um modelo treinado na aba **ML/Validate** para inferência em tempo real.
- Para documentação completa da API, consulte `docs/sphinx/`.